In [1]:
# configuração do SparkSession para usar Delta Lake


# ==============================
# IMPORTS
# ==============================
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from delta import *

# ==============================
# PATHS (AJUSTE CONFORME SEU AMBIENTE)
# ==============================
data_path = "data/vendas.csv"
delta_path = "data/delta/vendas"

# ==============================
# SPARK SESSION COM DELTA
# ==============================
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DeltaAtividade") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

26/04/21 18:21:23 WARN Utils: Your hostname, DESKTOP-8M7DIM9 resolves to a loopback address: 127.0.1.1; using 172.19.30.88 instead (on interface eth0)
26/04/21 18:21:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/vinicius/.cache/pypoetry/virtualenvs/trabalho-spark-engdados-6_wTpcsQ-py3.12/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/vinicius/.ivy2/cache
The jars for the packages stored in: /home/vinicius/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6ea8549c-8570-41bb-8c1e-0dda31f55b8b;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 173ms :: artifacts dl 6ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   | 

In [2]:
# Estrutura do Schema CSV


schema = StructType([
    StructField("id_venda", IntegerType(), True),
    StructField("produto", StringType(), True),
    StructField("quantidade", IntegerType(), True),
    StructField("valor_unitario", DoubleType(), True),
    StructField("data_venda", StringType(), True),
    StructField("regiao", StringType(), True)
])

In [3]:
# Leitura CSV

data_path = "/home/vinicius/trabalho-spark-engdados/data/vendas.csv"

df = spark.read.csv(
    data_path,
    header=True,
    schema=schema
)

In [4]:
# Escrita Delta lake

df.write.format("delta") \
    .mode("overwrite") \
    .save(delta_path)


In [5]:
# Criação de tabela Delta Lake


spark.sql(f"""
CREATE TABLE IF NOT EXISTS vendas_delta (
    id_venda INT,
    produto STRING,
    quantidade INT,
    valor_unitario DOUBLE,
    data_venda STRING,
    regiao STRING
)
USING DELTA
LOCATION '{delta_path}'
""")


DataFrame[]

In [ ]:
# INSERT


spark.sql("""
INSERT INTO vendas_delta VALUES
(2000, 'Mouse Gamer', 2, 150.0, '2024-03-01', 'Sul'),
(2001, 'Teclado Mecânico', 1, 300.0, '2024-03-01', 'Sudeste')
""")

In [ ]:
# UPDDATE

spark.sql("""
UPDATE vendas_delta
SET valor_unitario = 999.0
WHERE id_venda = 2000
""")

In [ ]:
# DELETE

spark.sql("""
DELETE FROM vendas_delta
WHERE id_venda = 2001
""")


In [6]:
# Consulta final para verificar alterações
 
spark.sql("SELECT * FROM vendas_delta").show()


26/04/21 18:21:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 7:====================================================>    (46 + 4) / 50]

+--------+-------+----------+--------------+----------+------+
|id_venda|produto|quantidade|valor_unitario|data_venda|regiao|
+--------+-------+----------+--------------+----------+------+
+--------+-------+----------+--------------+----------+------+



In [ ]:
 # TIME TRAVEL, Histórico de versões

spark.sql("DESCRIBE HISTORY vendas_delta").show()


In [7]:
 # TIME TRAVEL, Consulta versão anterior

spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load(delta_path) \
    .show()

+--------+----------------+----------+--------------+----------+--------+
|id_venda|         produto|quantidade|valor_unitario|data_venda|  regiao|
+--------+----------------+----------+--------------+----------+--------+
|       1|   Notebook Dell|         2|        3500.0|2024-01-15|     Sul|
|       2|  Mouse Logitech|         5|         120.0|2024-01-16|     Sul|
|       3|Teclado Mecânico|         1|         800.0|2024-01-17|Nordeste|
|       4|      Monitor LG|         1|        1500.0|2024-01-18| Sudeste|
|       5|       iPhone 15|         3|        5500.0|2024-01-19|   Norte|
|       6|  Fone Bluetooth|        10|         250.0|2024-01-20|     Sul|
+--------+----------------+----------+--------------+----------+--------+

